In [ ]:
import lancedb
from lancedb.pydantic import Vector
from lancedb.pydantic import LanceModel
from lancedb.embeddings import get_registry
from dotenv import load_dotenv
from google import genai
import pandas as pd
load_dotenv()

db = lancedb.connect(uri="vector_database")
model = get_registry().get("gemini-text").create(name="models/text-embedding-004")

In [30]:
class storyModel(LanceModel):
    doc_id: str
    filepath: str
    filename: str
    text: str = model.SourceField()
    vector: Vector(model.ndims()) = model.VectorField()
db2 = db.create_table("stories", schema=storyModel, exist_ok=True)

In [31]:
db.uri

'c:\\Users\\maxle\\Desktop\\Skola\\Ai_engineering_Max_Gustafsson\\code-alongs\\13_lancedb\\vector_database'

In [32]:
from pathlib import Path
import time
def ingest_data(table):    
    DATA_PATH = Path("data/sagor")
    files = list(DATA_PATH.glob("*.txt"))
    print(f"Found {len(files)} files to ingest.")
    for file in DATA_PATH.glob("*.txt"):
        with open(file, "r", encoding="utf-8") as f:
            text = f.read()

        doc_id = file.stem
        table.delete(f"doc_id = '{doc_id}'")
        table.add(
            [
                {
                    "doc_id": doc_id,
                    "filepath": str(file),
                    "filename": file.stem,
                    "text": text,
                }
            ]
        )
        print(table.to_pandas().shape)
        print(table.to_pandas()["filename"])
        time.sleep(30)

In [33]:
#def setup_vector_db(path):
 #   Path(path).mkdir(exist_ok = True)
  #  v_db = lancedb.connect(uri=path)
   # if "stories" in v_db.table_names():
    #    v_db.drop_table("stories")
    #v_db.create_table("stories", schema = storyModel, exist_ok = True)

    #return v_db

In [34]:
#VECTOR_PATH = Path("vector_database")
#vector_db = setup_vector_db(VECTOR_PATH)
#db.drop_table("stories")

#table = db.create_table("stories", schema=storyModel)

In [35]:
ingest_data(db["stories"])

Found 5 files to ingest.
(5, 5)
0     bockarna
1    kaninsaga
2      rödluva
3       snövit
4     askungen
Name: filename, dtype: object
(5, 5)
0    kaninsaga
1      rödluva
2       snövit
3     askungen
4     bockarna
Name: filename, dtype: object
(5, 5)
0      rödluva
1       snövit
2     askungen
3     bockarna
4    kaninsaga
Name: filename, dtype: object
(5, 5)
0       snövit
1     askungen
2     bockarna
3    kaninsaga
4      rödluva
Name: filename, dtype: object
(5, 5)
0     askungen
1     bockarna
2    kaninsaga
3      rödluva
4       snövit
Name: filename, dtype: object


In [36]:
db.open_table("stories").head()

pyarrow.Table
doc_id: string not null
filepath: string not null
filename: string not null
text: string not null
vector: fixed_size_list<item: float>[768]
  child 0, item: float
----
doc_id: [["askungen"],["bockarna"],...,["rödluva"],["snövit"]]
filepath: [["data\sagor\askungen.txt"],["data\sagor\bockarna.txt"],...,["data\sagor\rödluva.txt"],["data\sagor\snövit.txt"]]
filename: [["askungen"],["bockarna"],...,["rödluva"],["snövit"]]
text: [["Motivet för sagan återfinns så tidigt som under det första århundradet före Kristus, i en be (... 1510 chars omitted)"],["Sagan handlar om tre bockar, lilla bocken Bruse, mellanbocken Bruse och stora bocken Bruse. Dessa  (... 599 chars omitted)"],...,["En liten flicka som en gång fått en röd mössa av sin mormor skickas av sin mor med en korg med (... 953 chars omitted)"],["Snövit är en ung vacker flicka med läppar röda som rosor, hår svart som ebenholts och hy vit  (... 2581 chars omitted)"]]
vector: [[[-0.058746554,0.01563803,-0.0013810088,0.0387603

In [37]:
db["stories"].to_pandas()
db["stories"].to_pandas()["vector"]

0    [-0.058746554, 0.01563803, -0.0013810088, 0.03...
1    [-0.020389974, 0.027417121, -0.008008759, 0.01...
2    [-0.046967134, 0.0346248, -0.033076834, 0.0388...
3    [-0.044025704, -0.022275917, -0.046776593, 0.0...
4    [-0.028961252, 0.04120921, -0.030692138, 0.005...
Name: vector, dtype: object

In [41]:
query_text = ("Det var en gång en liten flicka som hette Rödluvan.")
query_text2 = ("Det var en gång en kung som bodde i ett stort slott.")
query_text3 = ("I en djup skog fanns det ett magiskt träd som kunde tala.")
query_text4 = ("Däremot blir han av med den fina rocken.")

#query_vector = model.generate_embeddings(query_text)
db["stories"].search(query_text).limit(5).to_pandas()

,doc_id,filepath,filename,text,vector,_distance
0,rödluva,data\sagor\rödluva.txt,rödluva,En liten flicka som en gång fått en röd mössa ...,"[-0.044025704, -0.022275917, -0.046776593, 0.0...",0.724885
1,snövit,data\sagor\snövit.txt,snövit,Snövit är en ung vacker flicka med läppar röda...,"[-0.028961252, 0.04120921, -0.030692138, 0.005...",0.810272
2,kaninsaga,data\sagor\kaninsaga.txt,kaninsaga,Pelle Kanin är ung och motsätter sig ofta förä...,"[-0.046967134, 0.0346248, -0.033076834, 0.0388...",0.876332
3,askungen,data\sagor\askungen.txt,askungen,Motivet för sagan återfinns så tidigt som unde...,"[-0.058746554, 0.01563803, -0.0013810088, 0.03...",0.941337
4,bockarna,data\sagor\bockarna.txt,bockarna,"Sagan handlar om tre bockar, lilla bocken Brus...","[-0.020389974, 0.027417121, -0.008008759, 0.01...",0.976191


In [39]:
db["stories"].search(query_text2).limit(5).to_pandas()

,doc_id,filepath,filename,text,vector,_distance
0,snövit,data\sagor\snövit.txt,snövit,Snövit är en ung vacker flicka med läppar röda...,"[-0.028961252, 0.04120921, -0.030692138, 0.005...",0.782647
1,askungen,data\sagor\askungen.txt,askungen,Motivet för sagan återfinns så tidigt som unde...,"[-0.058746554, 0.01563803, -0.0013810088, 0.03...",0.872364
2,kaninsaga,data\sagor\kaninsaga.txt,kaninsaga,Pelle Kanin är ung och motsätter sig ofta förä...,"[-0.046967134, 0.0346248, -0.033076834, 0.0388...",0.907138
3,rödluva,data\sagor\rödluva.txt,rödluva,En liten flicka som en gång fått en röd mössa ...,"[-0.044025704, -0.022275917, -0.046776593, 0.0...",0.918485
4,bockarna,data\sagor\bockarna.txt,bockarna,"Sagan handlar om tre bockar, lilla bocken Brus...","[-0.020389974, 0.027417121, -0.008008759, 0.01...",0.951863


In [40]:
db["stories"].search(query_text3).limit(5).to_pandas()

,doc_id,filepath,filename,text,vector,_distance
0,snövit,data\sagor\snövit.txt,snövit,Snövit är en ung vacker flicka med läppar röda...,"[-0.028961252, 0.04120921, -0.030692138, 0.005...",0.749808
1,kaninsaga,data\sagor\kaninsaga.txt,kaninsaga,Pelle Kanin är ung och motsätter sig ofta förä...,"[-0.046967134, 0.0346248, -0.033076834, 0.0388...",0.834238
2,rödluva,data\sagor\rödluva.txt,rödluva,En liten flicka som en gång fått en röd mössa ...,"[-0.044025704, -0.022275917, -0.046776593, 0.0...",0.919626
3,askungen,data\sagor\askungen.txt,askungen,Motivet för sagan återfinns så tidigt som unde...,"[-0.058746554, 0.01563803, -0.0013810088, 0.03...",0.953081
4,bockarna,data\sagor\bockarna.txt,bockarna,"Sagan handlar om tre bockar, lilla bocken Brus...","[-0.020389974, 0.027417121, -0.008008759, 0.01...",0.960582


In [43]:
db["stories"].search(query_text4).to_pandas()

,doc_id,filepath,filename,text,vector,_distance
0,kaninsaga,data\sagor\kaninsaga.txt,kaninsaga,Pelle Kanin är ung och motsätter sig ofta förä...,"[-0.046967134, 0.0346248, -0.033076834, 0.0388...",0.772948
1,snövit,data\sagor\snövit.txt,snövit,Snövit är en ung vacker flicka med läppar röda...,"[-0.028961252, 0.04120921, -0.030692138, 0.005...",0.937282
2,bockarna,data\sagor\bockarna.txt,bockarna,"Sagan handlar om tre bockar, lilla bocken Brus...","[-0.020389974, 0.027417121, -0.008008759, 0.01...",1.020062
3,rödluva,data\sagor\rödluva.txt,rödluva,En liten flicka som en gång fått en röd mössa ...,"[-0.044025704, -0.022275917, -0.046776593, 0.0...",1.040305
4,askungen,data\sagor\askungen.txt,askungen,Motivet för sagan återfinns så tidigt som unde...,"[-0.058746554, 0.01563803, -0.0013810088, 0.03...",1.063015


In [53]:
from google import genai
from dotenv import load_dotenv
load_dotenv()

client = genai.Client(api_key=GOOGLE_API_KEY)

llm = client.models.get("gemini-1.5-flash")
query_llm = model.generate_embeddings("Rödluvan")
dataquery = (db["stories"].search(query_llm).limit(1).to_pandas().iloc[0]["text"])
#print(dataquery)
sammanfittning = llm.generate_content(f"Sammanfatta följande text på ett kort och koncist sätt: {dataquery}")
print(sammanfittning.text)

NameError: name 'GOOGLE_API_KEY' is not defined